# The Temporal Dictionary Ensemble (TDE)

The Temporal Dictionary Ensemble [1] is a dictionary based classifier: each series
is transformed into a *bag of SFA words* (a histogram of discretised patterns), and
classification is 1-nearest-neighbour over those histograms using the histogram
intersection similarity. TDE brings together the strengths of the earlier
dictionary based classifiers - BOSS-style bags and ensembling, spatial pyramids to
retain coarse location information, bigrams, information gain binning - and selects
its ensemble members with a guided parameter search rather than a full grid.

TDE is the most accurate classifier of the dictionary based family on the UCR
archive, and it is one of the four components of HIVE-COTE 2.0 (HC2) [2], where it
represents that family alongside interval, shapelet and convolution based modules.
HC2 fits a full TDE on every problem, so TDE's runtime and memory directly shape
HC2's - one of the motivations for the implementation this notebook describes.

This notebook explains how a fitted TDE stores its bags of words and why, and
how the classifier uses them. It does not explain the SFA transform itself. It
uses private internals (`aeon.classification.dictionary_based._tde_sfa`) for
illustration - the goal is intuition about the mechanics of the algorithm.

#### References

[1] Middlehurst, M., Large, J., Cawley, G. and Bagnall, A. "The Temporal Dictionary
Ensemble (TDE) Classifier for Time Series Classification", in proceedings of the
European Conference on Machine Learning and Principles and Practice of Knowledge
Discovery in Databases, 2020.

[2] Middlehurst, M., Large, J., Flynn, M., Lines, J., Bostrom, A. and Bagnall, A.
"HIVE-COTE 2.0: a new meta ensemble for time series classification", Machine
Learning, 110:3211-3243, 2021.


## Motivation: from dictionaries to sorted arrays

Historically each bag was a numba typed `Dict` mapping word -> count. That is the
natural textbook structure, but it has two practical costs:

1. **Boundary costs.** Every dictionary operation performed from Python crosses the
   Python/numba boundary, so building bags and comparing them paid microseconds per
   *word* rather than per *series*. Typed Dicts also cannot be pickled, which forced
   conversion workarounds when saving models.
2. **Cache behaviour.** Comparing two histograms with hash lookups jumps around in
   memory. The comparison TDE needs (sum of minimum counts over shared words) does
   not actually need hashing at all.

The rewritten TDE stores **all bags of a model in four flat numpy arrays**, with each
case's bag stored as a contiguous run of rows, *sorted by key*. Three things follow:

1. **Pickling is trivial** - numpy arrays serialise natively, so all of the old
   save/load workarounds are gone.
2. **Speed comes from batching** - each hot loop is one long-running numba kernel
   call per model or ensemble member, not one call per case or per word.
3. **Exactness is preserved** - two sorted bags can be compared with a *merge
   intersection* that computes exactly the same sum of minimum counts the
   dictionary lookups did; the mechanics are shown below.


## The four arrays

A transform of `n` cases returns one tuple:

```
keys1   int64   the word itself (or a packed bigram)
keys2   int64   where the word came from: quadrant / channel tag (0 when unused)
counts  uint32  the histogram count for that (keys1, keys2) entry
offsets int64   case i's bag is rows offsets[i] : offsets[i+1]
```

Within each case's segment the rows are sorted lexicographically by
`(keys1, keys2)`. Let's build some bags and look at them.


In [ ]:
import numpy as np
import pandas as pd

from aeon.classification.dictionary_based._tde_sfa import _TDE_SFA

# six tiny series, two from each of three classes: sine waves of increasing
# frequency plus a little noise
rng = np.random.RandomState(0)
n_cases, n_timepoints = 6, 40
t = np.linspace(0, 2 * np.pi, n_timepoints)
X = np.zeros((n_cases, n_timepoints))
y = np.array([0, 0, 1, 1, 2, 2])
for i in range(n_cases):
    X[i] = np.sin((y[i] + 1) * t) + 0.05 * rng.randn(n_timepoints)

sfa = _TDE_SFA(word_length=4, window_size=8, levels=1, bigrams=False)
keys1, keys2, counts, offsets = sfa.fit_transform(X, y)

print("keys1  ", keys1)
print("keys2  ", keys2)
print("counts ", counts)
print("offsets", offsets)

## `offsets`: finding each case's bag

Bags have different sizes (word diversity and numerosity reduction vary per series),
so the rows of all cases are simply concatenated and `offsets` marks the boundaries.
It has `n_cases + 1` entries and case `i`'s bag is the slice
`offsets[i]:offsets[i+1]` of the other three arrays. For example, three cases with
bags of 4, 2 and 5 entries:

```
offsets = [0, 4, 6, 11]

row:      0    1    2    3  | 4    5  | 6    7    8    9    10
keys1:  ------- case 0 -----|- case 1 -|-------- case 2 --------
```

- the bag size of case `i` is `offsets[i+1] - offsets[i]`
- `offsets[0]` is always 0 and `offsets[-1]` is the total number of rows

If you know scipy sparse matrices, this is exactly the `indptr` array of CSR format:
the bags are one big sparse matrix (cases x possible words) stored row-compressed,
with the compound `(keys1, keys2)` pair playing the column index. It is also why the
nearest neighbour kernels can be handed a whole model at once - every bag of the
train and test sets arrives as a few flat arrays, sliced by integers inside numba,
with no per-case objects anywhere.


In [ ]:
for case in range(n_cases):
    size = offsets[case + 1] - offsets[case]
    print(
        f"case {case}: rows {offsets[case]:2d}:{offsets[case + 1]:2d}  "
        f"bag size {size}"
    )

With `levels=1` and univariate input, `keys2` is all zeros - the "where" part of the
key carries no information in this configuration (more on that below).

## Reading `keys1`: words as packed letters

The alphabet size is fixed at 4, so each letter of a word needs 2 bits and a word of
length `l` is an `l`-letter base-4 number packed into an int64, first letter in the
highest bits. Decoding is just shifts and masks:


In [ ]:
def decode_word(word, word_length=4):
    """Unpack an int64 SFA word into letters a-d, first letter first."""
    letters = []
    for i in range(word_length):
        shift = 2 * (word_length - 1 - i)
        letters.append("abcd"[(word >> shift) & 3])
    return "".join(letters)


def bag_frame(case):
    """Show one case's bag as a readable table."""
    s, e = offsets[case], offsets[case + 1]
    return pd.DataFrame(
        {
            "keys1": keys1[s:e],
            "word": [decode_word(w) for w in keys1[s:e]],
            "keys2": keys2[s:e],
            "count": counts[s:e],
        }
    )


bag_frame(0)

Each row reads as "this word occurred this many times in a sliding window of case 0".
The rows are sorted by `keys1` - that ordering is what makes the fast comparison
possible.

Compare a bag from each class - similar series share words, different classes mostly
do not:


In [ ]:
for case in (0, 2, 4):
    words = [decode_word(w) for w in keys1[offsets[case] : offsets[case + 1]]]
    print(f"case {case} (class {y[case]}): {words}")

## `keys2`: where a word came from

Each row of a bag is a triple: **which word** (`keys1`), **where it came from**
(`keys2`), and **how often it occurred** (`counts`). The "where" is coarse by
design - never the exact window position, only the pyramid quadrant and/or the
channel. `keys2` takes four forms depending on the configuration:

| configuration | `keys2` holds |
|---|---|
| univariate, `levels=1` | always 0 (unused) |
| univariate, `levels>1` | pyramid quadrant: 0 (level 0), 1-2 (level 1), 3-6 (level 2); `-1` for bigrams |
| multivariate, `levels=1` | the channel index the word was extracted from |
| multivariate, `levels>1` | `(quadrant << highest_channel_bit) | channel`; bigrams keep the `-1` quadrant, so their tags are negative |

The spatial pyramid (`levels>1`) is TDE's way of keeping some location information:
the same word occurring early in the series versus late is a *different histogram
entry*, weighted by pyramid level. So position must be part of the key:

In [ ]:
sfa3 = _TDE_SFA(word_length=4, window_size=8, levels=3, bigrams=True)
k1_3, k2_3, c_3, off_3 = sfa3.fit_transform(X, y)

print("distinct keys2 values:", sorted(set(k2_3.tolist())))

# one word from case 0 that appears in several quadrants
s, e = off_3[0], off_3[1]
frame = pd.DataFrame(
    {
        "word": [decode_word(w) for w in k1_3[s:e]],
        "keys2 (quadrant)": k2_3[s:e],
        "count": c_3[s:e],
    }
)
frame[frame["word"] == frame["word"].mode()[0]]

Quadrant 0 is the whole series (level 0), quadrants 1-2 are its halves (level 1) and
quadrants 3-6 its quarters (level 2). Note that `counts` is *weighted* here: a word
occurring `k` times in a level-`l` quadrant is stored with count `k * 2**l`, so the
more specific levels carry more weight in the similarity (you can see the same
occurrences counted 2 and 4 times in the table above). Occurrences are also counted
after numerosity reduction - consecutive repeats of the same word collapse to one.
The `-1` rows are bigrams (pairs of words one window apart), which are not assigned
to a quadrant and keep raw counts.

Why not pack the quadrant into `keys1` as well? No headroom: with bigrams a `keys1`
value can already use all 64 bits (a bigram of two 16-letter words is 2 x 32 bits),
so the "where" tag needs its own component and comparisons are lexicographic on
`(keys1, keys2)`.

## The comparison: merge intersection

The similarity between two bags is the sum of minimum counts over shared keys. With
both segments sorted, this is a two-pointer merge - no hashing, one linear pass:


In [ ]:
def merge_intersection(case_a, case_b):
    """Compute the histogram intersection of two cases, by hand."""
    i, a_end = offsets[case_a], offsets[case_a + 1]
    j, b_end = offsets[case_b], offsets[case_b + 1]
    sim = 0
    while i < a_end and j < b_end:
        ka, kb = (keys1[i], keys2[i]), (keys1[j], keys2[j])
        if ka == kb:
            sim += min(counts[i], counts[j])
            i += 1
            j += 1
        elif ka < kb:
            i += 1
        else:
            j += 1
    return sim


def dict_intersection(case_a, case_b):
    """Compute the same similarity the old way, via dictionaries."""
    s, e = offsets[case_a], offsets[case_a + 1]
    bag_a = {(keys1[r], keys2[r]): counts[r] for r in range(s, e)}
    s, e = offsets[case_b], offsets[case_b + 1]
    bag_b = {(keys1[r], keys2[r]): counts[r] for r in range(s, e)}
    return sum(min(v, bag_b.get(k, 0)) for k, v in bag_a.items())


for a, b in [(0, 1), (0, 2), (0, 4)]:
    assert merge_intersection(a, b) == dict_intersection(a, b)
    print(
        f"cases {a} (class {y[a]}) vs {b} (class {y[b]}): "
        f"similarity {merge_intersection(a, b)}"
    )

The merge and the dictionary versions agree exactly - the array representation
changes *how* the similarity is computed, never *what* is computed. Same-class pairs
score far higher than cross-class pairs, which is all the 1-nearest-neighbour
classifier needs.

Finally, check our by-hand merge against the numba kernel TDE actually uses:


In [ ]:
from aeon.classification.dictionary_based._tde_sfa import _histogram_intersection

for a, b in [(0, 1), (0, 2), (2, 3), (4, 5)]:
    kernel = _histogram_intersection(
        keys1, keys2, counts, offsets[a], offsets[a + 1], offsets[b], offsets[b + 1]
    )
    assert kernel == merge_intersection(a, b)
print("by-hand merge matches the library kernel on all pairs")

## From similarities to a classification

A single TDE member classifies a test case by 1-nearest-neighbour: build the test
case's bag, compare it against *every* training bag, and take the class of the most
similar one. Because all bags are flat arrays, the library computes every
test-vs-train similarity in one numba call, `nn_similarities_all`, which returns an
`(n_test, n_train)` integer matrix.

Let's reproduce a real `IndividualTDE` prediction by hand. First the library answer:


In [ ]:
from aeon.classification.dictionary_based import IndividualTDE

# three unseen test series, one per class
rng = np.random.RandomState(7)
X_test = np.zeros((3, n_timepoints))
for i in range(3):
    X_test[i] = np.sin((i + 1) * t) + 0.05 * rng.randn(n_timepoints)

clf = IndividualTDE(
    window_size=8, word_length=4, levels=1, bigrams=False, random_state=0
)
clf.fit(X[:, np.newaxis, :], y)  # classifier expects 3D (case, channel, time)
library_preds = clf.predict(X_test[:, np.newaxis, :])
print("library predictions:", library_preds)

Now by hand. The classifier's internal transform was fitted with the same
configuration and training data as our `sfa`, so `sfa` produces identical
bags and can stand in for it: transform the test series, compute the
similarity matrix with our merge from above, and take each row's most similar
training case:


In [ ]:
t_keys1, t_keys2, t_counts, t_offsets = sfa.transform(X_test)


def merge_between(a_arrays, a0, a1, b_arrays, b0, b1):
    """Compute the merge intersection between two bag sets' segments."""
    ak1, ak2, ac = a_arrays
    bk1, bk2, bc = b_arrays
    i, j, sim = a0, b0, 0
    while i < a1 and j < b1:
        ka, kb = (ak1[i], ak2[i]), (bk1[j], bk2[j])
        if ka == kb:
            sim += min(ac[i], bc[j])
            i += 1
            j += 1
        elif ka < kb:
            i += 1
        else:
            j += 1
    return sim


sims = np.zeros((3, n_cases), dtype=np.int64)
for test_case in range(3):
    for train_case in range(n_cases):
        sims[test_case, train_case] = merge_between(
            (t_keys1, t_keys2, t_counts),
            t_offsets[test_case],
            t_offsets[test_case + 1],
            (keys1, keys2, counts),
            offsets[train_case],
            offsets[train_case + 1],
        )

pd.DataFrame(sims, columns=[f"train {i} (class {c})" for i, c in enumerate(y)])

In [ ]:
from aeon.classification.dictionary_based._tde_sfa import nn_similarities_all

library_sims = nn_similarities_all(
    keys1, keys2, counts, offsets, t_keys1, t_keys2, t_counts, t_offsets
)
assert np.array_equal(sims, library_sims)

by_hand_preds = y[np.argmax(sims, axis=1)]
print("by-hand predictions:", by_hand_preds)
assert np.array_equal(by_hand_preds, library_preds.astype(int))

### Ties

`argmax` takes the *first* maximum, but the historical TDE rule is a coin flip per
tie: scanning a row left to right, an element *equal* to the running maximum takes
over with probability 0.5. Ties are not rare - bags of similar series share many
words - so this rule needed to survive the rewrite exactly, without paying for a
freshly seeded random generator per test case (that seeding briefly dominated
predict runtime).

Two observations make it fast:

1. The *running maximum itself* never depends on the coin flips - a tie changes
   which index is remembered, never the value. So `nn_first_max` can resolve
   tie-free rows entirely in numba and merely flag rows that had any tie event.
2. With an integer `random_state`, the scan builds a fresh, *identically seeded*
   generator per test case - so every case consumes the same draw sequence, and one
   precomputed pool of draws serves all rows (`nn_tie_break`).


In [ ]:
from sklearn.utils import check_random_state

from aeon.classification.dictionary_based._tde_sfa import nn_first_max, nn_tie_break

nn0, has_tie = nn_first_max(sims)
print("first-max index per test case:", nn0)
print("rows with tie events:         ", has_tie)

# the library path for an integer seed: one draw pool, all rows resolved in numba
draws = check_random_state(0).random(sims.shape[1])
nn_idx = nn_tie_break(sims, draws)

# replicate per-row seeded scans by hand and check they agree
for row in range(sims.shape[0]):
    rng = check_random_state(0)  # fresh generator per case, same seed
    best, chosen = -1, -1
    for j, s in enumerate(sims[row]):
        if s > best:
            best, chosen = s, j
        elif s == best and rng.random() < 0.5:
            chosen = j
    assert chosen == nn_idx[row]

print("pooled-draw kernel matches per-case seeded scans:", nn_idx)

Rows without tie events need no randomness at all, and for them
`nn_first_max`/`argmax`/`nn_tie_break` all agree. On real data most of the work is
the similarity matrix itself, which is why `predict` also threads cleanly: the
kernels are compiled with `nogil=True`, test cases are chunked across threads, and
an ensemble threads over its members - with results gathered in order, so
predictions are identical for any `n_jobs`.


## The ensemble: choosing members from a large parameter grid

A full TDE is an ensemble of up to 50 `IndividualTDE` members, each with its own
configuration of window size, word length, norm, pyramid levels and binning method.
The candidate grid is every combination of those - typically thousands - and
evaluating one candidate is expensive (fit on a 70% subsample, then estimate
accuracy by leave-one-out cross validation). So TDE samples `n_parameter_samples`
(default 250) candidates in two phases:

1. **Random**: the first `randomly_selected_params` (default 50) candidates are
   drawn uniformly from the grid.
2. **Guided**: after that, TDE fits a regressor mapping *parameters -> estimated
   accuracy* on everything evaluated so far, predicts the accuracy of every
   remaining candidate, and picks the best-looking one.

Let's look at the grid first:


In [ ]:
from aeon.classification.dictionary_based import TemporalDictionaryEnsemble

grid_probe = TemporalDictionaryEnsemble()
grid = grid_probe._unique_parameters(max_window=40, win_inc=1)
print(f"{len(grid)} candidates for a series of length 40, e.g.:")
pd.DataFrame(grid[:5], columns=["window_size", "word_length", "norm", "levels", "igb"])

The guided phase's regressor is kernel ridge regression on standardised parameters
(the TDE paper describes this step as a Gaussian process; the implementation has
always been kernel ridge). It used to be built from sklearn's `StandardScaler` +
`KernelRidge` on every iteration; profiling showed about a quarter of the total fit
time going into sklearn's per-call validation rather than the small solve, so the
computation is now ten lines of numpy in `_kernel_ridge_preds`. Fit a small
ensemble, then verify that helper against sklearn on the ensemble's own history:


In [ ]:
# a slightly larger toy problem so accuracy estimates are meaningful
rng = np.random.RandomState(1)
n_ens = 30
X_ens = np.zeros((n_ens, 1, n_timepoints))
y_ens = np.arange(n_ens) % 3
for i in range(n_ens):
    X_ens[i, 0] = np.sin((y_ens[i] + 1) * t) + 0.2 * rng.randn(n_timepoints)

ensemble = TemporalDictionaryEnsemble(
    n_parameter_samples=10,
    max_ensemble_size=5,
    randomly_selected_params=5,  # candidates 6-10 are regressor-guided
    random_state=0,
)
ensemble.fit(X_ens, y_ens)

pd.DataFrame(
    {
        "window_size": [m.window_size for m in ensemble.estimators_],
        "word_length": [m.word_length for m in ensemble.estimators_],
        "norm": [m.norm for m in ensemble.estimators_],
        "levels": [m.levels for m in ensemble.estimators_],
        "igb": [m.igb for m in ensemble.estimators_],
        "loocv accuracy": [m._accuracy for m in ensemble.estimators_],
        "weight": ensemble.weights_,
    }
)

Every evaluated candidate (kept or not) is recorded in `_prev_parameters_x` with its
accuracy in `_prev_parameters_y` - that is the regressor's training data. Members
are weighted by `accuracy ** 4`, so the ensemble's probability estimates lean
heavily on its best members:


In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import StandardScaler

from aeon.classification.dictionary_based._tde import _kernel_ridge_preds

# the weight rule
for m, w in zip(ensemble.estimators_, ensemble.weights_):
    assert w == m._accuracy**4

# the guided-selection regressor, checked against the sklearn original on the
# ensemble's real selection history
x_hist = np.array(ensemble._prev_parameters_x, dtype=np.float64)
y_hist = np.array(ensemble._prev_parameters_y, dtype=np.float64)
candidates = np.array(grid[:100], dtype=np.float64)

scaler = StandardScaler().fit(x_hist)
sk = KernelRidge(kernel="poly", degree=1)
sk.fit(scaler.transform(x_hist), y_hist)
expected = sk.predict(scaler.transform(candidates))

actual = _kernel_ridge_preds(x_hist, y_hist, candidates)
np.testing.assert_allclose(actual, expected, rtol=1e-8, atol=1e-10)

print("numpy kernel ridge matches sklearn; best-looking candidate of these 100:")
print(candidates[np.argmax(actual)], f"predicted accuracy {actual.max():.3f}")

Note how rough the regressor's predictions are with only ten evaluated candidates
behind it (they can even fall outside [0, 1] - it is a ridge regression, not a
probability model). That is fine for its job: it only has to *rank* candidates
better than random selection would, and it gets more history with every candidate
evaluated. With the default 250 parameter samples, 200 picks are guided.

The selection loop itself is unchanged from the original algorithm: fit each chosen
candidate on a fresh 70% subsample (re-drawn until it contains at least two
classes), estimate its accuracy, and either grow the ensemble or replace the current
worst member if the newcomer beats it. The accuracy estimate is the expensive step,
and it has its own story - the leave-one-out search exploits the symmetry of the
histogram intersection to halve its work.


## Scoring a member: leave-one-out cross validation

Each candidate's quality score is its leave-one-out 1NN accuracy on its own
training subsample: for every case, find the nearest neighbour *among the other
cases* and check the class matches. That is inherently O(n^2) histogram
intersections, and it runs once per candidate - with default settings, 250 times
per fit - which made it the single largest cost in the ensemble.

The optimisation is the symmetry of the histogram intersection:
`sim(i, j) == sim(j, i)`, but predicting case `i` and case `j` separately computes
the pair twice. `loocv_train_acc` instead fills an `n x n` similarity matrix
upper-triangle-only, mirroring each value, so every pair is merged exactly once -
half the work. (The matrix is int32 and a subsample is typically a few hundred
cases, so it is kilobytes; above 4096 cases the code falls back to a per-case
search rather than allocate it.)

Reproduce it by hand on our six training bags:


In [ ]:
from aeon.classification.dictionary_based._tde_sfa import loocv_train_acc

# by hand: for each case, the first maximum over all *other* cases
loocv_preds = np.empty(n_cases, dtype=np.int64)
for i in range(n_cases):
    best, nn = -1, -1
    for j in range(n_cases):
        if j == i:
            continue
        s = merge_intersection(i, j)
        if s > best:
            best, nn = s, j
    loocv_preds[i] = nn
loocv_correct = int(np.sum(y[loocv_preds] == y))

n_done, correct, preds = loocv_train_acc(
    keys1, keys2, counts, offsets, y.astype(np.int64), 0
)

assert n_done == n_cases
assert np.array_equal(preds, loocv_preds)
assert correct == loocv_correct
print("nearest neighbour of each case:", preds)
print(f"LOOCV accuracy: {correct}/{n_cases}")

Each case's nearest neighbour is its same-class partner - the toy problem is easy.
(Unlike predict, LOOCV ties have always been resolved deterministically by first
maximum, so this kernel needs no randomness.)

### Early abandoning

The ensemble does not always need the exact accuracy. Once it is full, a new
candidate only matters if it beats the current worst member, so the scan is handed
`required_correct` and stops the moment that target becomes arithmetically
unreachable - before each row it checks
`correct_so_far + rows_remaining < required_correct`. An abandoned candidate
reports `-1`, which the ensemble treats as "discard":


In [ ]:
# relabel the cases so every nearest neighbour now has a *different* class: the
# first prediction is wrong, so a demand for all six correct becomes unreachable
# after one row and the scan stops there
y_bad = np.array([0, 1, 2, 0, 1, 2], dtype=np.int64)
n_done, correct, _ = loocv_train_acc(keys1, keys2, counts, offsets, y_bad, n_cases)
print(f"abandoned after {n_done} of {n_cases} rows, reported accuracy {correct}")
assert correct == -1 and 0 < n_done < n_cases

Preserving this exactly (rather than always computing everything) matters beyond
speed: the recorded accuracy feeds the kernel ridge regressor that guides later
parameter choices, so an abandoned candidate must report `-1` at the same point it
always did.

Two reuse notes: the per-case predictions returned by the kernel are exactly what
`fit_predict_proba` needs for train estimates, so no second pass is made; and
multivariate channel selection (next) reuses this same kernel to score each
channel.


## Multivariate series: channel selection and bag merging

For multivariate input each member selects its own channels, in three steps:

1. **Score every channel cheaply.** A per-channel transform is fitted, and the
   channel's usefulness is estimated with the LOOCV kernel above (`required_correct=0`,
   so nothing is abandoned) on *reduced*
   bags - built from the few non-overlapping binning windows instead of every
   sliding window, so a fast proxy.
2. **Keep the good ones.** Channels within `channel_threshold` (default 85%) of the
   best channel's score survive, capped at `max_channels` randomly. Different members
   end up with different channel subsets, so channel selection is itself ensembled.
3. **Merge the survivors' bags.** Each kept channel's full bags are built and
   merged into one bag per case, with the channel recorded in `keys2` so words from
   different channels never combine.

Build a three-channel toy problem where channel 0 carries the class signal,
channel 1 is pure noise and channel 2 is a noisier copy of the signal, then
reproduce the channel selection by hand:

In [ ]:
rng = np.random.RandomState(3)
n_mv = 12
X_mv = np.zeros((n_mv, 3, n_timepoints))
y_mv = (np.arange(n_mv) % 3).astype(np.int64)
for i in range(n_mv):
    signal = np.sin((y_mv[i] + 1) * t)
    X_mv[i, 0] = signal + 0.05 * rng.randn(n_timepoints)  # clean signal
    X_mv[i, 1] = rng.randn(n_timepoints)  # pure noise
    X_mv[i, 2] = signal + 0.4 * rng.randn(n_timepoints)  # noisier signal

# by hand: score each channel by LOOCV on its reduced (binning-window) bags
channel_scores = []
for channel in range(3):
    ch_sfa = _TDE_SFA(
        word_length=4,
        window_size=8,
        levels=1,
        bigrams=False,
        keep_binning_dft=True,
    )
    ch_sfa.fit(X_mv[:, channel, :], y_mv)
    _, correct, _ = loocv_train_acc(*ch_sfa.binning_bags(), y_mv, 0)
    channel_scores.append(correct)

max_score = max(channel_scores)
by_hand_channels = [ch for ch in range(3) if channel_scores[ch] >= max_score * 0.85]
print("channel scores:", channel_scores, "-> selected channels:", by_hand_channels)

clf_mv = IndividualTDE(
    window_size=8, word_length=4, levels=1, bigrams=False, random_state=0
)
clf_mv.fit(X_mv, y_mv)
assert clf_mv._channels == by_hand_channels
print("classifier selected:", clf_mv._channels)

The pure-noise channel scores far below 85% of the clean channel and is dropped.

The merged bags use the `keys2` forms from the table above: with `levels=1` the tag
is simply the channel index, so slicing the combined arrays by `keys2` recovers
each channel's own bag exactly. The merge itself never re-sorts anything - each
channel's bags are already sorted and the channel tag preserves order, so the
combined bag is a k-way merge of sorted streams (one numba call, no aggregation
needed since keys from different channels can never be equal):


In [ ]:
ck1, ck2, cc, coff = clf_mv._transformed_data
print("distinct keys2 values in the combined bags:", sorted(set(ck2.tolist())))

# recover channel 0's bag for case 0 from the combined arrays and check it
# against a fresh transform of that channel alone
s, e = coff[0], coff[1]
in_channel_0 = ck2[s:e] == 0
recovered = list(zip(ck1[s:e][in_channel_0], cc[s:e][in_channel_0]))

ch_sfa = _TDE_SFA(word_length=4, window_size=8, levels=1, bigrams=False)
k1c, k2c, cc_c, off_c = ch_sfa.fit_transform(X_mv[:, 0, :], y_mv)
direct = list(zip(k1c[off_c[0] : off_c[1]], cc_c[off_c[0] : off_c[1]]))

assert recovered == direct
print("channel 0 rows of the combined bag match a direct channel-0 transform")

After fitting, the transformers' cached Fourier arrays are dropped
(`_clear_transformer_fit_cache`), so a fitted model keeps only breakpoints and
bags. Predict follows the identical path - per-channel transform, the same k-way
merge, and the combined bags flow through exactly the `nn_similarities_all`
machinery shown earlier, with no multivariate special-casing.


## Wrap-up: how this is verified, and what it bought

The array representation replaced a dictionary-based implementation, so the
essential question is whether it is still *the same classifier*. That is
established in three layers, coarsest to finest:

1. **Exact unit-level parity.** The test suite asserts the new transform produces
   bags exactly equal to the generic SFA transformer across representative
   configurations, and every kernel met in this notebook (`loocv_train_acc`,
   `nn_tie_break`, `combine_channel_bags`, the merge intersection) has hand-checkable
   unit tests - several of which are the same by-hand reproductions performed
   above.
2. **Behavioural invariants.** `n_jobs` never changes predictions; pickled models
   predict identically after a round trip; the numpy kernel ridge matches sklearn
   to numerical precision; and `IndividualTDE` still reproduces the probabilities
   archived in aeon's expected-results tests.
3. **Statistical equivalence.** Bit-identity with the old implementation was
   deliberately not the goal (float-level differences exist, most fundamentally in
   members where `word_length >= window_size`, whose trailing letters are rounding
   noise in either implementation). Benchmarks on the UCR archive and EEG
   classification problems show accuracies not significantly different from the
   previous version.

What the representation bought, measured against the previous implementation on
like-for-like benchmarks:

| | fit | predict |
|---|---|---|
| array bags + batched kernels | 10-23x faster (growing with dataset size) | 4-11x faster |
| + pooled tie-break draws | - | a further ~2.4x, single threaded |
| + threading (`n_jobs=8`) | fit is single threaded by design | ~3x more |

The remaining runtime floor is the merge intersections themselves - the actual
O(n^2) similarity work of the algorithm. Fit stays sequential because guided
parameter selection is inherently so: each choice depends on every accuracy
estimated before it.